# 02: Clean, subset, and document

We preserve the raw response and create an analysis-ready CSV. Cleaning is a set of visible choices that we make.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
csv_path = root/'data/processed/daily_streamflow.csv'
streamflow = pd.read_csv(csv_path, parse_dates=['date'])
streamflow.head()

,date,discharge_cfs,qualifier
0,2024-05-01,255.0,A
1,2024-05-02,220.0,A
2,2024-05-03,168.0,A
3,2024-05-04,137.0,A
4,2024-05-05,118.0,A


## Inspect the CSV

A CSV has no built-in knowledge of dates, units, or missing-value rules. `parse_dates` tells pandas that `date` is a date; inspect types before doing calculations.

In [7]:
streamflow.info()
streamflow.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           153 non-null    datetime64[ns]
 1   discharge_cfs  153 non-null    float64       
 2   qualifier      153 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 3.7+ KB


date             0
discharge_cfs    0
qualifier        0
dtype: int64

## Select only the fields needed

Selecting columns makes an analysis intent clearer and avoids carrying unrelated fields into later work.

In [8]:
analysis_columns = ['date', 'discharge_cfs', 'qualifier']
subset = streamflow.loc[:, analysis_columns].copy()
subset.head()

,date,discharge_cfs,qualifier
0,2024-05-01,255.0,A
1,2024-05-02,220.0,A
2,2024-05-03,168.0,A
3,2024-05-04,137.0,A
4,2024-05-05,118.0,A


## Subset a time window

This exemplar keeps the full May–September period. Try a shorter window to see how an explicit date condition changes the table. Always document a subset rule in a real project.

In [9]:
june_to_august = subset.loc[subset['date'].between('2024-06-01', '2024-08-31')].copy()
print('Rows in June–August subset:', len(june_to_august))
june_to_august.head()

Rows in June–August subset: 92


,date,discharge_cfs,qualifier
31,2024-06-01,368.0,A
32,2024-06-02,426.0,A
33,2024-06-03,440.0,A
34,2024-06-04,526.0,A
35,2024-06-05,673.0,A


## Check and drop missing discharge values

Here we drop rows only when the value required for this analysis (`discharge_cfs`) is missing. We do **not** silently fill values, and the raw download remains untouched.

In [10]:
print('Missing discharge before:', june_to_august['discharge_cfs'].isna().sum())
analysis_ready = june_to_august.dropna(subset=['discharge_cfs']).reset_index(drop=True)
print('Rows after dropna:', len(analysis_ready))
analysis_ready.head()

Missing discharge before: 0
Rows after dropna: 92


,date,discharge_cfs,qualifier
0,2024-06-01,368.0,A
1,2024-06-02,426.0,A
2,2024-06-03,440.0,A
3,2024-06-04,526.0,A
4,2024-06-05,673.0,A


## Reuse the project transformation

The script applies the same documented rule to the full period and writes `data/processed/daily_streamflow.csv`. This is preferable to relying on a one-off notebook state.

In [11]:
from src.processing import make_processed_data
processed = make_processed_data()
processed.head()

,date,discharge_cfs,qualifier
0,2024-05-01,255.0,A
1,2024-05-02,220.0,A
2,2024-05-03,168.0,A
3,2024-05-04,137.0,A
4,2024-05-05,118.0,A


## Tiny data dictionary

| Field | Meaning |
| --- | --- |
| `date` | Calendar day of the daily observation. |
| `discharge_cfs` | Daily mean streamflow in cubic feet per second. |
| `qualifier` | USGS qualification code(s); check source documentation before filtering. |

### Checkpoint

What is the difference between a reproducible subset and an arbitrary deletion? What information should you record before dropping values?